### RAG pipelines- Data Ingestion to Vector DB piplines 

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader , PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

c:\Users\manas_978n0hd\OneDrive\Desktop\Full stack app development\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in directory"""

    all_documents =[]
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print (pdf_files)

    print(f" Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"Processing : {pdf_file.name} ")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            for doc in documents:   
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")

    return all_documents

all_pdf_documents = process_all_pdfs("../data")

[WindowsPath('../data/pdf_files/pdf1.pdf'), WindowsPath('../data/pdf_files/pdf2.pdf'), WindowsPath('../data/pdf_files/pdf3.pdf'), WindowsPath('../data/pdf_files/pdf4.pdf')]
 Found 4 PDF files to process
Processing : pdf1.pdf 
  ✓ Loaded 11 pages
Processing : pdf2.pdf 
  ✓ Loaded 11 pages
Processing : pdf3.pdf 
  ✓ Loaded 11 pages
Processing : pdf4.pdf 
  ✓ Loaded 11 pages


In [ ]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-04-24T01:06:25+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-04-24T01:06:25+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf_files\\pdf1.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': 'pdf1.pdf', 'file_type': 'pdf'}, page_content='Sample PDF Document 1\nThis is paragraph 1 of Sample PDF Document 1. This is paragraph 1 of Sample PDF Document 1.\nThis is paragraph 1 of Sample PDF Document 1. This is paragraph 1 of Sample PDF Document 1.\nThis is paragraph 1 of Sample PDF Document 1. This is paragraph 1 of Sample PDF Document 1.\nThis is paragraph 1 of Sample PDF Document 1. This is paragraph 1 of Sample PDF Document 1.\nThis is paragraph 1 of Sample PDF Document 1. This is paragraph 1 of Sample PDF Document 1.\nThis is paragraph 1 of Sample PDF Document 1. This is 

In [ ]:
### Text splitting get into chunks

def split_documents(documents, chunk_size= 1000, chunk_overlap=200):
    """Split Documents into smaller chunks for better RAG performance."""
    text_splitter= RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n","\n"," ",""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print("\n Example of Chunks")
        print(f"Content1: {split_docs[0].page_content[:200]}")
        print(f"Metadata: {split_docs[0].metadata}")
        
    return split_docs


In [ ]:
chunks = split_documents(all_pdf_documents)

Split 44 documents into 260 chunks

 Example of Chunks
Content1: Sample PDF Document 1
This is paragraph 1 of Sample PDF Document 1. This is paragraph 1 of Sample PDF Document 1.
This is paragraph 1 of Sample PDF Document 1. This is paragraph 1 of Sample PDF Docume
Metadata: {'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-04-24T01:06:25+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-04-24T01:06:25+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf_files\\pdf1.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': 'pdf1.pdf', 'file_type': 'pdf'}


### Embedding and vector store DB

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    """Handles document embedding generations using SentenceTransformer"""

    def __init__(self, model_name: str= "all-miniLM-L6-V2"):
        """Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the Sentence Transformer model"""
        try:
            print(f"loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape(len(texts),embedding_dimensions)
        """

        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddinggs for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embedding with shape: {embeddings.shape}")
        return embeddings

In [ ]:
embedding_manager = EmbeddingManager()
embedding_manager 

loading embedding model: all-miniLM-L6-V2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8581.57it/s]


Model loaded successfully. Embedding dimension: 384


### Vectore Store